In [1]:
from peft import LoraConfig, TaskType, get_peft_model
from transformers import BertForSequenceClassification, BertTokenizer
from src.helper_fn import get_dataset, get_trainable_parameters, compute_metrics
from transformers import TrainingArguments, Trainer
import wandb

/Users/meninderpurewal/miniconda3/envs/hf2/lib/python3.11/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


'NoneType' object has no attribute 'cadam32bit_grad_fp32'


In [31]:
tokenizer = BertTokenizer.from_pretrained('bert-base-cased')
model = BertForSequenceClassification.from_pretrained('bert-base-cased', num_labels=2)
get_trainable_parameters(model)

Some weights of the model checkpoint at bert-base-cased were not used when initializing BertForSequenceClassification: ['cls.predictions.transform.LayerNorm.weight', 'cls.predictions.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initi

'trainable params: 108311810 || all params: 108311810 || trainable%: 100.00'

In [3]:
train_dataset, test_dataset = get_dataset(tokenizer, cap_rows_train=1_000, cap_rows_test=500)

Found cached dataset yelp_polarity (/Users/meninderpurewal/.cache/huggingface/datasets/yelp_polarity/plain_text/1.0.0/14f90415c754f47cf9087eadac25823a395fef4400c7903c5897f55cfaaa6f61)


  0%|          | 0/2 [00:00<?, ?it/s]

2023-10-01 13:10:14,526 - src.logger - INFO - Capping dataset rows to 1000 train and 500 test
Loading cached processed dataset at /Users/meninderpurewal/.cache/huggingface/datasets/yelp_polarity/plain_text/1.0.0/14f90415c754f47cf9087eadac25823a395fef4400c7903c5897f55cfaaa6f61/cache-aad40350f2e86711.arrow
Loading cached processed dataset at /Users/meninderpurewal/.cache/huggingface/datasets/yelp_polarity/plain_text/1.0.0/14f90415c754f47cf9087eadac25823a395fef4400c7903c5897f55cfaaa6f61/cache-bf813e794de0b26e.arrow
2023-10-01 13:10:14,717 - src.logger - INFO - Train shape: (1000, 2), Test shape: (500, 2)


In [4]:
train_dataset1 = train_dataset.with_format("torch", )
test_dataset1 = test_dataset.with_format("torch", )

In [5]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=4,
    lora_alpha=1,
    lora_dropout=0.1,
    #target_modules=["query", "value"],
)
lora_model = get_peft_model(model, lora_config)
get_trainable_parameters(lora_model)


'trainable params: 150532 || all params: 108460804 || trainable%: 0.14'

In [33]:
model.bert.encoder.layer[0].attention.self

BertSelfAttention(
  (query): Linear(in_features=768, out_features=768, bias=True)
  (key): Linear(in_features=768, out_features=768, bias=True)
  (value): Linear(in_features=768, out_features=768, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
)

In [54]:
model.bert.encoder.layer[0]

BertLayer(
  (attention): BertAttention(
    (self): BertSelfAttention(
      (query): Linear(in_features=768, out_features=768, bias=True)
      (key): Linear(in_features=768, out_features=768, bias=True)
      (value): Linear(in_features=768, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (output): BertSelfOutput(
      (dense): Linear(in_features=768, out_features=768, bias=True)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (intermediate): BertIntermediate(
    (dense): Linear(in_features=768, out_features=3072, bias=True)
    (intermediate_act_fn): GELUActivation()
  )
  (output): BertOutput(
    (dense): Linear(in_features=3072, out_features=768, bias=True)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
)

In [6]:
training_args = TrainingArguments(
    output_dir='lora_debug',
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    learning_rate=5e-5,
    warmup_ratio=0.1,
    evaluation_strategy="steps",
    save_steps=800,
    eval_steps=400,
    save_total_limit=1,
    use_mps_device='mps',
    logging_dir='lora_debug/logs',
    optim='adamw_torch',
    report_to='wandb',
    run_name='batch16'
    )

In [7]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset1,
    eval_dataset=test_dataset1,
    compute_metrics=compute_metrics
)

In [8]:
trainer.train()
wandb.finish()

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: meninder-purewal. Use `wandb login --relogin` to force relogin


  0%|          | 0/63 [00:00<?, ?it/s]